# 12. ML Topic Classification - Prototype / GPT Mini Fallback

11번 노트북에서 생성된 샘플 주제분류 결과와 topic prototype을 기준으로, 아직 분류되지 않은 원천 memo를 embedding 후 prototype similarity로 1차 분류합니다.

- similarity >= `ml_classification.auto_accept_threshold`: ML 자동 확정
- similarity < threshold: `llm_fallback_queue`에 적재 후 GPT mini fallback 대상으로 관리


In [ ]:
import sys
import importlib

from pyspark.sql import functions as F

PROJECT_ROOT = "/Workspace/Users/jungryo.lee@lge.com/prj_TV_voc"
SRC_ROOT = f"{PROJECT_ROOT}/src"

if SRC_ROOT not in sys.path:
    sys.path.append(SRC_ROOT)

import common.config_loader as config_loader
import ml.unclassified_embedding as unclassified_embedding
import ml.topic_ml_classifier as topic_ml_classifier
import pipeline.run_ml_topic_classification as run_ml_topic_classification

importlib.reload(config_loader)
importlib.reload(unclassified_embedding)
importlib.reload(topic_ml_classifier)
importlib.reload(run_ml_topic_classification)

from common.config_loader import load_config, get_output_table, get_reference_table
from pipeline.run_ml_topic_classification import run_ml_topic_classification

config = load_config(f"{PROJECT_ROOT}/config/settings_intellytics.yaml")

print("settings =", config["path"]["settings"])
print("embedding_model =", config["ml_classification"]["embedding_model"])
print("fallback_model =", config["ml_classification"]["fallback_model_key"])


In [ ]:
# 완료된 샘플 주제분류 그룹 확인
classification_detail_table = get_output_table(config, "classification_detail")
topic_prototype_table = get_output_table(config, "topic_prototype")
category_mapping_table = get_reference_table(config, "category_mapping_table")

completed_group_df = spark.sql(f"""
SELECT
  COALESCE(m.cate_1_depth_kor, c.cate_1_depth) AS cate_1_depth_kor,
  COALESCE(m.cate_2_depth_kor, c.cate_2_depth) AS cate_2_depth_kor,
  c.cate_1_depth,
  c.cate_2_depth,
  c.sc_measurement,
  COUNT(*) AS classified_sample_cnt,
  COUNT(DISTINCT c.memo_id) AS classified_distinct_memo_cnt,
  SUM(CASE WHEN c.pred_topic_type = 'topic' THEN 1 ELSE 0 END) AS topic_cnt,
  SUM(CASE WHEN c.pred_topic_type = 'overall' THEN 1 ELSE 0 END) AS overall_cnt,
  SUM(CASE WHEN c.pred_topic_type = 'others' THEN 1 ELSE 0 END) AS others_cnt,
  MAX(c.created_at) AS last_classified_at
FROM {classification_detail_table} c
LEFT JOIN {category_mapping_table} m
  ON c.cate_1_depth = m.cate_1_depth
 AND c.cate_2_depth = m.cate_2_depth
 AND m.is_lifestyle = 'N'
WHERE c.prompt_version = '{config['version']['prompt_version']}'
  AND c.taxonomy_version = '{config['version']['taxonomy_version']}'
GROUP BY
  COALESCE(m.cate_1_depth_kor, c.cate_1_depth),
  COALESCE(m.cate_2_depth_kor, c.cate_2_depth),
  c.cate_1_depth,
  c.cate_2_depth,
  c.sc_measurement
ORDER BY cate_1_depth_kor, cate_2_depth_kor, c.sc_measurement
""")

display(completed_group_df)


In [ ]:
# 실행 옵션
# 처음 검증 시에는 그룹별 샘플 수를 100~200 정도로 두고, 결과 확인 후 확대하세요.
# LIMIT_ROWS는 전체 상한, LIMIT_ROWS_PER_GROUP은 카테고리/감성 그룹별 상한입니다.
LIMIT_ROWS = None
LIMIT_ROWS_PER_GROUP = 200
RUN_EMBEDDING = True
RUN_CLASSIFICATION = True
# 비용 안전장치: fallback queue 규모를 확인한 뒤 True로 바꾸세요.
RUN_GPT_MINI_FALLBACK = False
FALLBACK_LIMIT_ROWS = 50
SKIP_EXISTING = True

# 이전 12번 테스트 결과를 지우고 균형 샘플을 새로 보고 싶을 때만 True로 바꾸세요.
RESET_PREVIOUS_12_TEST_OUTPUTS = False

if RESET_PREVIOUS_12_TEST_OUTPUTS:
    for table_key in ["memo_embedding_unclassified", "ml_classification_detail", "llm_fallback_queue"]:
        table_name = get_output_table(config, table_key)
        if spark.catalog.tableExists(table_name):
            print("reset table =", table_name)
            spark.sql(f"""
            DELETE FROM {table_name}
            WHERE prompt_version = '{config['version']['prompt_version']}'
              AND taxonomy_version = '{config['version']['taxonomy_version']}'
            """)

result = run_ml_topic_classification(
    spark,
    config,
    run_embedding=RUN_EMBEDDING,
    run_classification=RUN_CLASSIFICATION,
    run_llm_fallback=RUN_GPT_MINI_FALLBACK,
    limit_rows=LIMIT_ROWS,
    limit_rows_per_group=LIMIT_ROWS_PER_GROUP,
    fallback_limit_rows=FALLBACK_LIMIT_ROWS,
    skip_existing=SKIP_EXISTING,
)

result


In [ ]:
# ML 자동분류 결과 확인
ml_table = get_output_table(config, "ml_classification_detail")

ml_df = spark.table(ml_table).where(F.col("prompt_version") == config["version"]["prompt_version"])

display(
    ml_df.groupBy(
        "cate_1_depth",
        "cate_2_depth",
        "sc_measurement",
        "classification_stage",
        "pred_topic",
    )
    .agg(
        F.count("*").alias("memo_cnt"),
        F.avg("confidence_score").alias("avg_confidence"),
    )
    .orderBy("cate_1_depth", "cate_2_depth", "sc_measurement", F.desc("memo_cnt"))
)


In [ ]:
# ML fallback row와 GPT mini fallback queue 동기화 검증
ml_table = get_output_table(config, "ml_classification_detail")
queue_table = get_output_table(config, "llm_fallback_queue")

fallback_required_df = (
    spark.table(ml_table)
    .where(F.col("prompt_version") == config["version"]["prompt_version"])
    .where(F.col("taxonomy_version") == config["version"]["taxonomy_version"])
    .where(F.col("classification_stage") == "embedding_prototype_llm_fallback")
)

pending_queue_df = (
    spark.table(queue_table)
    .where(F.col("prompt_version") == config["version"]["prompt_version"])
    .where(F.col("taxonomy_version") == config["version"]["taxonomy_version"])
    .where(F.col("status") == "pending")
)

sync_summary_df = spark.createDataFrame([
    {
        "fallback_required_cnt": fallback_required_df.count(),
        "fallback_required_distinct_memo_id_cnt": fallback_required_df.select("memo_id").dropDuplicates().count(),
        "pending_queue_cnt": pending_queue_df.count(),
        "pending_queue_distinct_memo_id_cnt": pending_queue_df.select("memo_id").dropDuplicates().count(),
    }
])

display(
    sync_summary_df.withColumn(
        "queue_synced",
        (F.col("fallback_required_cnt") == F.col("pending_queue_cnt"))
        & (F.col("fallback_required_distinct_memo_id_cnt") == F.col("pending_queue_distinct_memo_id_cnt")),
    )
)


In [ ]:
# GPT mini fallback 후보 확인
queue_table = get_output_table(config, "llm_fallback_queue")

queue_df = spark.table(queue_table).where(F.col("prompt_version") == config["version"]["prompt_version"])

display(
    queue_df.groupBy("cate_1_depth", "cate_2_depth", "sc_measurement", "status")
    .agg(
        F.count("*").alias("fallback_cnt"),
        F.avg("prototype_confidence_score").alias("avg_prototype_confidence"),
    )
    .orderBy(F.desc("fallback_cnt"))
)

display(
    queue_df.select(
        "cate_1_depth",
        "cate_2_depth",
        "sc_measurement",
        "memo",
        "prototype_confidence_score",
        "candidate_topics_json",
        "fallback_model_key",
        "fallback_model_version",
    ).orderBy(F.asc("prototype_confidence_score")).limit(50)
)
